# 02. 정책 스코어링 및 정책 키워드 대응도 분석

온통청년 정책 데이터만으로 지역별 정책 공급 점수, 일자리 문제 유형별 정책 키워드 대응도, 지역별 정책 대응 부족 분야를 계산하였습니다.

주의: 해당 결과는 실제 채용공고 자료와 결합한 미스매치 점수가 아니라, 정책 텍스트를 기반으로 한 일자리 문제 대응도 분석입니다.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

candidate_paths = [OUTPUT_DIR / "policy_preprocessed_solo.csv", OUTPUT_DIR / "policy_preprocessed.csv", BASE_DIR / "youth_policies_categorized.csv"]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
print("입력 파일:", DATA_PATH)
if DATA_PATH is None:
    raise FileNotFoundError("전처리 파일이 없습니다. 01_policy_preprocessing_solo.ipynb를 먼저 실행하세요.")

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print("데이터 크기:", df.shape)
display(df.head(3))

입력 파일: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project\outputs\policy_preprocessed_solo.csv
데이터 크기: (5470, 50)


,지역,조회_zipCd,정책ID,정책명,정책키워드,정책설명,정책지원내용,정책대분류,정책중분류,자동분류,대표분류,신청기간,사업기간,지원대상,나이조건,소득조건,신청방법,제출서류,주관기관,운영기관,신청URL,참고URL1,참고URL2,최초등록일시,최종수정일시,수집페이지,수집출처,정책ID_정리,정책명_정리,정책명_정제,정책키워드_정제,정책설명_정제,정책지원내용_정제,지원대상_정제,신청방법_정제,주관기관_정제,운영기관_정제,분석텍스트,대표분류_개선,매칭분류_전체,매칭분류수,_최종수정일시_dt,시작일_추정,종료일_추정,상시성정책,신청가능_추정,신청URL_존재,참고URL_존재,신청방법_존재,설명길이
0,강원,"51000,42000",20260605005400113228,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정)","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정부기여금 지원)",금융･복지･문화,취약계층 및 금융지원,복지,복지,20260622 ~ 20261231,20260622 ~ 20261231,"19세~34세 / 연령제한:N 0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상 0011009 0013010 0049010 0055003",19세~34세 / 연령제한:N,"0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상",ㅇ 취급은행 모바일앱을 통해 매월 비대면 신청 가능 ㅇ 2026년 6월 출시 예정,NaN,금융위원회,한국고용정보원,NaN,https://www.kinfa.or.kr/financialProduct/youthFutureSavings.do,https://blog.naver.com/blogfsc/224302863400,2026-06-05 18:06:49,2026-06-10 14:05:31,1,온통청년_OPEN_API_getPlcy,20260605005400113228,청년미래적금,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정)","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정부기여금 지원)","19세~34세 / 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상",ㅇ 취급은행 모바일앱을 통해 매월 비대면 신청 가능 ㅇ 2026년 6월 출시 예정,금융위원회,한국고용정보원,"청년미래적금 보조금 청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(2026년 6월 22일 출시 예정) 은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 ...",창업지원,"복지, 일자리, 창업지원, 참여 프로그램",4,2026-06-10 14:05:31,2026-06-22 00:00:00,2026-12-31 00:00:00,0,1,0,1,1,288
1,강원,"51000,42000",20250114005400210229,스마트 모빌리티 창업캠프사업,교육지원,"미래모빌리티 분야 창업을 희망하는 청년을 대상으로 국내 자동차업계 마이스터들의 강연, 멘토링, 카운슬링을 지원하여 창업아이디어를 구체화하고 창업 준비를 위한 각종 프로그램 운영","1. (사전프로그램) 기업가정신 함양, 멘토링, 카운슬링 2. (집합프로그램) - 자동차산업 전망, 창업 핵심기반, 기업가정신 등 교육지원 - 창업아이디어 도출･평가, 개발 제품 설계･평가 지원 등 실습 - 대한민국 미래모빌리티엑스포 연계, 창...","일자리,일자리","취업,창업","일자리, 직무교육, 창업지원, 참여 프로그램",일자리,20260401 ~ 20260528,20260301 ~ 20261231,"0세~0세 / 연령제한:Y 0043001 0 0 모빌리티 분야(관련) 전공 대학(원)생 0011003,0011004,0011005 0013010 0049005,0049006,0049007,0049008 0055003",0세~0세 / 연령제한:Y,0043001 0 0,공고에 따라 지원서 작성 후 이메일 제출(신청기간 연장~5/28),대학생 스마트 모빌리티 창업캠프 지원서,미래모빌리티과,대구광역시 미래혁신성장실,https://www.kaae.kr/,https://www.kaae.kr/,NaN,2025-01-14 11:00:54,2026-05-29 14:57:57,5,온통청년_OPEN_API_getPlcy,20250114005400210229,스마트 모빌리티 창업캠프사업,스마트 모빌리티 창업캠프사업,교육지원,"미래모빌리티 분야 창업을 희망하는 청년을 대상으로 국내 자동차업계 마이스터들의 강연, 멘토링, 카운슬링을 지원하여 창업아이디어를 구체화하고 창업 준비를 위한 각종 프로그램 운영","1. (사전프로그램) 기업가정신 함양, 멘토링, 카운슬링 2. (집합프로그램) - 자동차산업 전망, 창업 핵심기반, 기업가정신 등 교육지원 - 창업아이디어 도출･평가, 개발 제품 설계･평가 지원 등 실습 - 대한민국 미래모빌리티엑스포 연계, 창...","0세~0세 / 모빌리티 분야(관련) 전공 대학(원)생 , , , , ,",공고에 따라 지원서 작성 후 이메일 제출(신청기간 연장~5/28),미래모빌리티과,대구광역시 미래혁신성장실,"스마트 모빌리티 창업캠프사업 교육지원 미래모빌리티 분야 창업을 희망하는 청년을 대상으로 국내 자동차업계 마이스터들의 강연, 멘토링, 카운슬링을 지원하여 창업아이디어를 구체화하고 창업 준비를 위한 각종 프로그램 운영 1. (사전프로그램) 기업가정...",창업지원,"창업지원, 직무교육, 일자리, 참여 프로그램",4,2026-05-29 14:57:57,2026-03-01 00:00:00,2026-12-31 00:00:00,0,1,1,1,1,400
2,강원,"51000,42000",20250714005400111227,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,"교육지원,맞춤형상담서비스,장기미취업청년",산림자원을 활용 기반의 산림 문제 해결형 사업 공모전 개최 및 실증 과정을 통한 청년 창업 경험 지원,ㅇ 공모전명 : 2026 청년 林팩트 창업 아이디어 챌린지 ㅇ 주요내용 : 산림자원을 활용 기반의 산림 문제 해결형 사업 공모전 개최 및 실증 과정을 통한 청년 창업 경험 지원 ㅇ 일 정 : 2026.6.22.(월) ~ 6.23(화) / [참가...,"일자리,일자리","취업,창업","일자리, 직무교육, 창업지원, 복지, 참여 프로그램",일자리,20260520 ~ 20260603,20260622 ~ 20260623,19세~39세 / 연령제한:N 0043001 0 0 산림창업에 관심있는 청년(만19세 ~ 39세) 누구나 0011009 0013010 0049010 0055003,19세~39세 / 연령제한:N,0043001 0 0,NaN,1. 신청사이트 URL을 통하여 참가신청서 작성 2. 한국임업진흥원 누리집(www.kofpi.or.kr)의 ‘알림/홍보’ - ‘공지사항’ 메뉴 '청년 임팩트 창업 아이디어 챌린지' 게시판의 참가신청 큐알(QR) 코드 또는 링크(https://d...,산림청,한국임업진흥원,https://docs.google.com/forms/u/0/d/1-QTMyaKxeBcQvtNYkMlNIkhtZ-K1lxyVEBMDn9WsAuo/viewform?edit_requested=true,www.kofpi.or.kr,NaN,2025-07-14 22:19:33,2026-05-29 11:08:25,2,온통청년_OPEN_API_getPlcy,20250714005400111227,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,"교육지원,맞춤형상담서비스,장기미취업청년",산림자원을 활용 기반의 산림 문제 해결형 사업 공모전 개최 

## 1. 일자리 문제 유형 키워드 사전과 가중치

혼자 진행하는 소프로젝트이므로 채용공고 결과 파일을 입력받지 않고, 채용공고 분석에서 흔히 쓰는 문제 유형을 기준 사전으로 정의합니다.

In [2]:
ISSUE_KEYWORDS = {
    "고용기회": ["취업", "채용", "구직", "일자리", "고용", "면접", "인턴", "근로", "일경험", "취업연계", "알선"],
    "임금소득": ["임금", "급여", "월급", "소득", "수당", "생활비", "지원금", "장려금", "교통비", "식비", "자산", "적금", "통장", "금융"],
    "직무성장": ["교육", "훈련", "직무", "자격증", "역량", "멘토링", "컨설팅", "인재양성", "디지털", "AI", "코딩", "실습"],
    "워라밸환경": ["워라밸", "근로환경", "복지", "휴가", "유연근무", "건강", "심리", "상담", "문화", "근속", "휴식"],
    "주거안정": ["주거", "월세", "전세", "임대", "주택", "보증금", "기숙사", "청년주택", "전월세", "주거비"],
    "창업생태계": ["창업", "스타트업", "사업화", "예비창업", "창업자", "창업공간", "사업자", "소상공인", "기업가"],
    "참여관계": ["참여", "네트워크", "동아리", "커뮤니티", "공모전", "청년활동", "위원회", "서포터즈", "교류", "소통"],
}

ISSUE_WEIGHTS = {
    "고용기회": 0.22,
    "임금소득": 0.18,
    "직무성장": 0.18,
    "워라밸환경": 0.14,
    "주거안정": 0.12,
    "창업생태계": 0.10,
    "참여관계": 0.06,
}
print("문제유형 가중치 합:", sum(ISSUE_WEIGHTS.values()))

문제유형 가중치 합: 1.0


## 2. 정책별 문제 유형 대응 여부 계산

In [3]:
if "분석텍스트" not in df.columns:
    text_cols = [c for c in ["정책명", "정책키워드", "정책설명", "정책지원내용", "지원대상", "신청방법"] if c in df.columns]
    df["분석텍스트"] = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)

def contains_any(text, keywords):
    text = str(text).lower()
    return int(any(str(kw).lower() in text for kw in keywords))

for issue, keywords in ISSUE_KEYWORDS.items():
    df[f"문제대응_{issue}"] = df["분석텍스트"].apply(lambda x: contains_any(x, keywords))

issue_cols = [f"문제대응_{issue}" for issue in ISSUE_KEYWORDS]
df["문제대응_총개수"] = df[issue_cols].sum(axis=1)
df["일자리문제관련정책"] = (df["문제대응_총개수"] > 0).astype(int)

weighted_score = np.zeros(len(df))
for issue, weight in ISSUE_WEIGHTS.items():
    weighted_score += df[f"문제대응_{issue}"] * weight

df["정책별_키워드대응점수_raw"] = weighted_score
df["정책별_키워드대응점수_100"] = (df["정책별_키워드대응점수_raw"] / max(ISSUE_WEIGHTS.values()) * 100).clip(0, 100).round(2)

issue_match_summary = pd.DataFrame({
    "문제유형": list(ISSUE_KEYWORDS.keys()),
    "매칭정책수": [int(df[f"문제대응_{issue}"].sum()) for issue in ISSUE_KEYWORDS],
})
issue_match_summary["전체대비비율"] = (issue_match_summary["매칭정책수"] / len(df)).round(4)
display(issue_match_summary)
display(df[["지역", "정책명", "문제대응_총개수", "정책별_키워드대응점수_100"]].head())

,문제유형,매칭정책수,전체대비비율
0,고용기회,2407,0.4400
1,임금소득,1697,0.3102
2,직무성장,3324,0.6077
3,워라밸환경,2300,0.4205
4,주거안정,621,0.1135
5,창업생태계,1323,0.2419
6,참여관계,1919,0.3508


,지역,정책명,문제대응_총개수,정책별_키워드대응점수_100
0,강원,청년미래적금,4,100.0
1,강원,스마트 모빌리티 창업캠프사업,2,100.0
2,강원,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,5,100.0
3,강원,(농식품부) 농식품 바우처,3,100.0
4,강원,삼척형 청년인턴 지원사업,2,100.0


## 3. 지역별 정책 공급 종합점수

정책수, 관련정책비율, 키워드대응도, 분류다양성, 신청가능성, 정보접근성을 정규화한 뒤 가중합합니다.

In [4]:
def minmax(series, neutral=50):
    s = pd.to_numeric(series, errors="coerce").fillna(0)
    if s.max() == s.min():
        return pd.Series([neutral] * len(s), index=s.index)
    return (s - s.min()) / (s.max() - s.min()) * 100

category_col = "대표분류_개선" if "대표분류_개선" in df.columns else ("대표분류" if "대표분류" in df.columns else "자동분류")

region_score = df.groupby("지역").agg(
    정책수=("정책명", "count"),
    일자리문제관련정책수=("일자리문제관련정책", "sum"),
    평균키워드대응점수=("정책별_키워드대응점수_100", "mean"),
    평균문제대응개수=("문제대응_총개수", "mean"),
    신청가능비율=("신청가능_추정", "mean") if "신청가능_추정" in df.columns else ("일자리문제관련정책", "mean"),
    신청URL보유비율=("신청URL_존재", "mean") if "신청URL_존재" in df.columns else ("일자리문제관련정책", "mean"),
    신청방법보유비율=("신청방법_존재", "mean") if "신청방법_존재" in df.columns else ("일자리문제관련정책", "mean"),
).reset_index()
region_score["일자리문제관련정책비율"] = region_score["일자리문제관련정책수"] / region_score["정책수"]
region_score["정보접근성비율"] = (region_score["신청URL보유비율"] + region_score["신청방법보유비율"]) / 2

category_count = df[df[category_col].ne("기타")].groupby("지역")[category_col].nunique().reset_index(name="분류개수")
max_category_count = max(1, len([c for c in df[category_col].dropna().unique() if c != "기타"]))
category_count["분류다양성점수"] = category_count["분류개수"] / max_category_count * 100
region_score = region_score.merge(category_count, on="지역", how="left")
region_score["분류다양성점수"] = region_score["분류다양성점수"].fillna(0)

region_score["정책수점수"] = minmax(np.log1p(region_score["정책수"]))
region_score["관련정책비율점수"] = minmax(region_score["일자리문제관련정책비율"])
region_score["키워드대응도점수"] = minmax(region_score["평균키워드대응점수"])
region_score["신청가능성점수"] = minmax(region_score["신청가능비율"])
region_score["정보접근성점수"] = minmax(region_score["정보접근성비율"])

SCORING_WEIGHTS = {
    "정책수점수": 0.25,
    "관련정책비율점수": 0.25,
    "키워드대응도점수": 0.25,
    "분류다양성점수": 0.10,
    "신청가능성점수": 0.10,
    "정보접근성점수": 0.05,
}
region_score["정책공급종합점수"] = sum(region_score[col] * weight for col, weight in SCORING_WEIGHTS.items()).round(2)
region_score["정책대응부족도"] = (100 - region_score["정책공급종합점수"]).round(2)
region_score = region_score.sort_values("정책공급종합점수", ascending=False).reset_index(drop=True)
display(region_score)

,지역,정책수,일자리문제관련정책수,평균키워드대응점수,평균문제대응개수,신청가능비율,신청URL보유비율,신청방법보유비율,일자리문제관련정책비율,정보접근성비율,분류개수,분류다양성점수,정책수점수,관련정책비율점수,키워드대응도점수,신청가능성점수,정보접근성점수,정책공급종합점수,정책대응부족도
0,경북,539,532,93.321002,2.625232,0.569573,0.387755,0.589981,0.987013,0.488868,6,100.0,36.545291,100.000000,100.000000,50.313907,34.553283,75.90,24.10
1,제주,608,596,92.494128,2.509868,0.560855,0.486842,0.659539,0.980263,0.573191,6,100.0,64.714649,77.077109,80.778401,45.624896,100.000000,75.21,24.79
2,경남,584,572,92.652654,2.547945,0.476027,0.349315,0.539384,0.979452,0.444349,6,100.0,55.295961,74.322546,84.463511,0.000000,0.000000,63.52,36.48
3,충북,498,488,92.990221,2.560241,0.508032,0.429719,0.672691,0.979920,0.551205,6,100.0,18.047542,75.910629,92.310627,17.213831,82.935642,62.44,37.56
4,경기,515,505,92.021320,2.466019,0.578641,0.431068,0.633010,0.980583,0.532039,6,100.0,25.895353,78.161699,69.787454,55.190883,68.060010,62.38,37.62
5,서울,461,453,93.078373,2.605206,0.527115,0.436009,0.609544,0.982646,0.522777,6,100.0,0.000000,85.170836,94.359824,27.477587,60.871127,60.67,39.33
6,강원,489,477,92.006053,2.507157,0.521472,0.423313,0.664622,0.975460,0.543967,6,100.0,13.783876,60.765666,69.432551,24.442713,77.318244,52.31,47.69
7,전남,535,520,91.164093,2.446729,0.555140,0.362617,0.605607,0.971963,0.484112,6,100.0,34.803586,48.887891,49.860258,42.551027,30.861828,49.19,50.81
8,전북,534,519,90.602809,2.402622,0.573034,0.357678,0.593633,0.971910,0.475655,6,100.0,34.366129,48.709582,36.812574,52.175106,24.298166,46.40,53.60
9,충남,707,677,89.019208,2.260255,0.661952,0.333805,0.613861,0.957567,0.473833,6,100.0,100.000000,0.000000,0.000000,100.000000,22.883767,46.14,53.86


## 4. 지역별 문제 유형 대응도와 부족 분야

In [5]:
issue_region_rows = []
for region, g in df.groupby("지역"):
    total = len(g)
    for issue in ISSUE_KEYWORDS:
        count = int(g[f"문제대응_{issue}"].sum())
        issue_region_rows.append({
            "지역": region,
            "문제유형": issue,
            "대응정책수": count,
            "지역정책수": total,
            "대응비율": count / total if total else 0,
            "문제유형가중치": ISSUE_WEIGHTS[issue],
        })
issue_region = pd.DataFrame(issue_region_rows)
issue_region["대응비율정규화"] = issue_region.groupby("문제유형")["대응비율"].transform(lambda s: minmax(s))
issue_region["부족도"] = (1 - issue_region["대응비율"]) * issue_region["문제유형가중치"]

issue_pivot = issue_region.pivot(index="지역", columns="문제유형", values="대응비율").fillna(0).reset_index()
weak_issue = issue_region.sort_values(["지역", "부족도"], ascending=[True, False]).groupby("지역").head(2).groupby("지역").agg(정책대응부족분야=("문제유형", lambda x: ", ".join(x))).reset_index()
region_score = region_score.merge(weak_issue, on="지역", how="left")

def level_label(score):
    if score >= 75: return "높음"
    if score >= 60: return "보통 이상"
    if score >= 45: return "보통"
    return "낮음"
region_score["정책공급수준"] = region_score["정책공급종합점수"].apply(level_label)
region_score["해석"] = region_score.apply(lambda r: f"{r['지역']}은 정책공급종합점수 {r['정책공급종합점수']:.1f}점으로 '{r['정책공급수준']}' 수준이며, 보완 필요 분야는 {r.get('정책대응부족분야', '')}로 나타남.", axis=1)

interpretation_table = region_score[["지역", "정책수", "일자리문제관련정책비율", "평균키워드대응점수", "정책공급종합점수", "정책대응부족도", "정책대응부족분야", "해석"]].copy()
display(issue_pivot)
display(interpretation_table)

문제유형,지역,고용기회,워라밸환경,임금소득,주거안정,직무성장,참여관계,창업생태계
0,강원,0.464213,0.421268,0.296524,0.087935,0.609407,0.370143,0.257669
1,경기,0.450485,0.417476,0.297087,0.104854,0.617476,0.351456,0.227184
2,경남,0.438356,0.422945,0.318493,0.140411,0.635274,0.345890,0.246575
3,경북,0.467532,0.447124,0.332096,0.126160,0.604824,0.359926,0.287570
4,서울,0.464208,0.431670,0.347072,0.108460,0.624729,0.386117,0.242950
5,전남,0.418692,0.386916,0.306542,0.112150,0.613084,0.345794,0.263551
6,전북,0.425094,0.417603,0.301498,0.110487,0.567416,0.340824,0.239700
7,제주,0.442434,0.429276,0.322368,0.118421,0.608553,0.356908,0.231908
8,충남,0.390382,0.393211,0.263083,0.108911,0.596888,0.306931,0.200849
9,충북,0.461847,0.447791,0.335341,0.112450,0.602410,0.365462,0.234940


,지역,정책수,일자리문제관련정책비율,평균키워드대응점수,정책공급종합점수,정책대응부족도,정책대응부족분야,해석
0,경북,539,0.987013,93.321002,75.90,24.10,"임금소득, 고용기회","경북은 정책공급종합점수 75.9점으로 '높음' 수준이며, 보완 필요 분야는 임금소득, 고용기회로 나타남."
1,제주,608,0.980263,92.494128,75.21,24.79,"고용기회, 임금소득","제주은 정책공급종합점수 75.2점으로 '높음' 수준이며, 보완 필요 분야는 고용기회, 임금소득로 나타남."
2,경남,584,0.979452,92.652654,63.52,36.48,"고용기회, 임금소득","경남은 정책공급종합점수 63.5점으로 '보통 이상' 수준이며, 보완 필요 분야는 고용기회, 임금소득로 나타남."
3,충북,498,0.979920,92.990221,62.44,37.56,"임금소득, 고용기회","충북은 정책공급종합점수 62.4점으로 '보통 이상' 수준이며, 보완 필요 분야는 임금소득, 고용기회로 나타남."
4,경기,515,0.980583,92.021320,62.38,37.62,"임금소득, 고용기회","경기은 정책공급종합점수 62.4점으로 '보통 이상' 수준이며, 보완 필요 분야는 임금소득, 고용기회로 나타남."
5,서울,461,0.982646,93.078373,60.67,39.33,"고용기회, 임금소득","서울은 정책공급종합점수 60.7점으로 '보통 이상' 수준이며, 보완 필요 분야는 고용기회, 임금소득로 나타남."
6,강원,489,0.975460,92.006053,52.31,47.69,"임금소득, 고용기회","강원은 정책공급종합점수 52.3점으로 '보통' 수준이며, 보완 필요 분야는 임금소득, 고용기회로 나타남."
7,전남,535,0.971963,91.164093,49.19,50.81,"고용기회, 임금소득","전남은 정책공급종합점수 49.2점으로 '보통' 수준이며, 보완 필요 분야는 고용기회, 임금소득로 나타남."
8,전북,534,0.971910,90.602809,46.40,53.60,"고용기회, 임금소득","전북은 정책공급종합점수 46.4점으로 '보통' 수준이며, 보완 필요 분야는 고용기회, 임금소득로 나타남."
9,충남,707,0.957567,89.019208,46.14,53.86,"고용기회, 임금소득","충남은 정책공급종합점수 46.1점으로 '보통' 수준이며, 보완 필요 분야는 고용기회, 임금소득로 나타남."


## 5. 저장

In [6]:
weights_table = pd.DataFrame({
    "점수구성요소": list(SCORING_WEIGHTS.keys()),
    "가중치": list(SCORING_WEIGHTS.values()),
    "설명": [
        "지역별 정책 수를 로그 변환 후 Min-Max 정규화",
        "전체 정책 중 일자리 문제 유형 키워드에 대응하는 정책 비율",
        "정책별 문제유형 키워드 가중 매칭 평균",
        "일자리·교육·주거·창업·복지·참여 등 분류 다양성",
        "신청가능 또는 상시성 정책 비율",
        "신청 URL 및 신청방법 정보 보유 정도",
    ]
})
issue_weights_table = pd.DataFrame({"문제유형": list(ISSUE_WEIGHTS.keys()), "문제유형가중치": list(ISSUE_WEIGHTS.values()), "대표키워드": [", ".join(ISSUE_KEYWORDS[k][:8]) for k in ISSUE_WEIGHTS]})
weight_export = pd.concat([
    weights_table.assign(구분="정책공급종합점수"),
    issue_weights_table.rename(columns={"문제유형": "점수구성요소", "문제유형가중치": "가중치", "대표키워드": "설명"}).assign(구분="문제유형가중치")
], ignore_index=True)

df.to_csv(OUTPUT_DIR / "policy_scored_solo.csv", index=False, encoding="utf-8-sig")
region_score.to_csv(OUTPUT_DIR / "policy_region_score_solo.csv", index=False, encoding="utf-8-sig")
issue_region.to_csv(OUTPUT_DIR / "policy_issue_region_coverage_solo.csv", index=False, encoding="utf-8-sig")
issue_pivot.to_csv(OUTPUT_DIR / "policy_issue_region_pivot_solo.csv", index=False, encoding="utf-8-sig")
interpretation_table.to_csv(OUTPUT_DIR / "policy_region_interpretation_solo.csv", index=False, encoding="utf-8-sig")
weight_export.to_csv(OUTPUT_DIR / "policy_scoring_weights_solo.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", OUTPUT_DIR)

저장 완료: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project\outputs


## 보고서용 스코어링 방법론 요약

본 소프로젝트는 외부 채용공고 스코어링 파일을 입력받지 않고, 온통청년 정책 데이터의 정책명·정책설명·정책키워드·지원내용을 기반으로 정책 키워드 대응도를 분석하였다. 채용공고 분석에서 일반적으로 확인되는 일자리 문제를 고용기회, 임금소득, 직무성장, 워라밸환경, 주거안정, 창업생태계, 참여관계의 7개 유형으로 정의하고, 각 유형별 대표 키워드 사전을 구축하였다. 지역별 정책 공급 종합점수는 정책 수, 일자리 문제 관련 정책 비율, 키워드 대응도, 분류 다양성, 신청 가능성, 정보 접근성을 Min-Max 정규화한 뒤 가중합하여 산출하였다. 단, 본 점수는 실제 채용공고 원자료와 직접 결합한 미스매치 점수가 아니라 정책 데이터 내부에서 산출한 정책 키워드 대응도 기반의 상대 비교 지표이다.